In [150]:
from openai import OpenAI
import os

openai_client = OpenAI(
    # api_key=os.getenv('GROQ_API_KEY'),
    # base_url='https://api.groq.com/openai/v1'
)
# model = 'llama-3.3-70b-versatile'
model = 'gpt-5.4-mini'

In [131]:
import os

def see_file_tree(root_dir: str = ".") -> list[str]:
    """List files and directories in the current project.

    Args:
        root_dir: Directory to list. Defaults to the current directory.
    """
    if root_dir == "":
        root_dir = "."
    
    tree = []

    for dirpath, dirnames, filenames in os.walk(root_dir):
        dirnames[:] = [d for d in dirnames if d not in {
            ".git", ".venv", "__pycache__", "uv.lock", ".ipynb_checkpoints"}]

        for name in sorted(dirnames + filenames):
            full_path = os.path.join(dirpath, name)
            tree.append(os.path.relpath(full_path, root_dir))

    return tree

In [132]:
see_file_tree_description = {
    "type": "function",
    "name": "see_file_tree",
    "description": "List files and directories in the current project.",
    "parameters": {
        "type": "object",
        "properties": {
            "root_dir": {
                "type": "string",
                "description": "Directory to list. Defaults to the current directory. Use '.' for current directory",
            }
        },
        "required": [],
        "additionalProperties": False,
    },
}

In [133]:
def read_file(filepath: str) -> str:
    """Read the contents of a file in the current project.

    Args:
        filepath: Path to the file to read.
    """
    try:
        if filepath == 'uv.lock':
            return "you're not allowed to read this file"

        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: file '{filepath}' not found."

In [99]:
read_file_description = {
    "type": "function",
    "name": "read_file",
    "description": "Read the contents of a file in the current project.",
    "parameters": {
        "type": "object",
        "properties": {
            "filepath": {
                "type": "string",
                "description": "Path to the file to read.",
            }
        },
        "required": ["filepath"],
        "additionalProperties": False,
    },
}

In [100]:
import json

In [101]:
model = 'openai/gpt-oss-120b'

In [147]:
system_prompt = """
You are a helpful coding assistant.
Don't make any assumptions about the nature of the project, 
use all available tools to first explore and then give your answer

When a user wants to create code, you don't show code to the user,
use use the write tool to create code files
"""

user_prompt = "What dependencies does this project have?"

chat_messages = [
    {"role": "developer", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [123]:
request_number = 1

while True:
    print(f'request #{request_number}...')
    response = openai_client.responses.create(
        model=model,
        input=chat_messages,
        tools=[see_file_tree_description, read_file_description]
    )

    has_tool_calls = False
    
    chat_messages.extend(response.output)
    
    for item in response.output:
        if item.type == 'function_call':
            has_tool_calls = True 
    
            f_name = item.name
            args = json.loads(item.arguments)
            print(f'tool_call: {f_name}({args})')
    
            if f_name == 'see_file_tree':
                result = see_file_tree(**args)
            elif f_name == 'read_file':
                result = read_file(**args)
            else:
                raise Exception(f'unknown function {f_name}')
    
            chat_messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result),
            })
        elif item.type == 'message':
            print(item.content[0].text)

    if not has_tool_calls:
        break

    request_number = request_number + 1

request #1...
tool_call: see_file_tree({'root_dir': '.'})
request #2...
tool_call: read_file({'filepath': 'pyproject.toml'})
request #3...
tool_call: read_file({'filepath': 'uv.lock'})
request #4...
The project’s declared dependencies are listed in **`pyproject.toml`** under the `[project]` table:

```toml
[project]
name = "workshop"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.14"

dependencies = [
    "jupyter>=1.1.1",
    "openai>=2.31.0",
]
```

**Summary of dependencies**

| Dependency | Minimum version |
|------------|-----------------|
| `jupyter`  | 1.1.1 |
| `openai`   | 2.31.0 |

Additionally, the project requires Python **3.14 or newer**. No other dependency files (e.g., `requirements.txt`) are present in the repository.


In [124]:
!uv add toyaikit

Resolved 114 packages in 366ms                                       
Prepared 1 package in 29ms                                               
Installed 4 packages in 22ms                                
 + anthropic==0.94.1
 + docstring-parser==0.18.0
 + genai-prices==0.0.56
 + toyaikit==0.0.9


In [125]:
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner

In [134]:
tools_obj = Tools()
tools_obj.add_tool(see_file_tree)
tools_obj.add_tool(read_file)

In [135]:
tools_obj.get_tools()

[{'type': 'function',
  'name': 'see_file_tree',
  'description': 'List files and directories in the current project.\n\nArgs:\n    root_dir: Directory to list. Defaults to the current directory.',
  'parameters': {'type': 'object',
   'properties': {'root_dir': {'type': 'string',
     'description': 'root_dir parameter'}},
   'required': [],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'read_file',
  'description': 'Read the contents of a file in the current project.\n\nArgs:\n    filepath: Path to the file to read.',
  'parameters': {'type': 'object',
   'properties': {'filepath': {'type': 'string',
     'description': 'filepath parameter'}},
   'required': ['filepath'],
   'additionalProperties': False}}]

In [151]:
chat_interface = IPythonChatInterface()
llm_client = OpenAIClient(client=openai_client, model=model)

runner = OpenAIResponsesRunner(
    tools=tools_obj,
    developer_prompt=system_prompt,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [152]:
result = runner.run()

You: stop


Chat ended.


In [138]:
import os
from pathlib import Path

class ProjectTools:
    def __init__(self, project_dir: Path):
        self.project_dir = project_dir

    def see_file_tree(self, root_dir: str = ".") -> list[str]:
        """List files and directories in the project.

        Args:
            root_dir: Directory to list relative to the project root.
        """
        abs_root = self.project_dir / root_dir
        tree = []

        for dirpath, dirnames, filenames in os.walk(abs_root):
            dirnames[:] = [d for d in dirnames if d not in {".git", ".venv", "__pycache__"}]

            for name in sorted(dirnames + filenames):
                full_path = os.path.join(dirpath, name)
                tree.append(os.path.relpath(full_path, self.project_dir))

        return tree

    def read_file(self, filepath: str) -> str:
        """Read the contents of a file relative to the project root.

        Args:
            filepath: Path to the file to read.
        """
        abs_path = self.project_dir / filepath
        try:
            with open(abs_path, "r", encoding="utf-8") as f:
                return f.read()
        except FileNotFoundError:
            return f"Error: file '{filepath}' not found."

    def search_in_files(self, search_term: str) -> list[str]:
        """Search for a string in project files.

        Args:
            search_term: Text to search for.
        """
        matches = []

        for dirpath, dirnames, filenames in os.walk(self.project_dir):
            dirnames[:] = [d for d in dirnames if d not in {".git", ".venv", "__pycache__"}]

            for filename in filenames:
                abs_path = Path(dirpath) / filename

                try:
                    content = abs_path.read_text(encoding="utf-8")
                except Exception:
                    continue

                if search_term in content:
                    matches.append(str(abs_path.relative_to(self.project_dir)))

        return matches

In [139]:
from pathlib import Path
project_tools = ProjectTools(Path('.'))

In [140]:
project_tools.see_file_tree()

['.env',
 '.gitignore',
 '.ipynb_checkpoints',
 '.python-version',
 'README.md',
 'Untitled.ipynb',
 'main.py',
 'pyproject.toml',
 'uv.lock',
 '.ipynb_checkpoints/Untitled-checkpoint.ipynb']

In [141]:
project_tools.read_file('README.md')

''

In [142]:
!head README.md

In [153]:
tools_obj = Tools()
tools_obj.add_tools(project_tools)

In [156]:
import tools
agent_tools = tools.AgentTools(Path("."))

tools_obj = Tools()
tools_obj.add_tools(agent_tools)

In [157]:
chat_interface = IPythonChatInterface()
llm_client = OpenAIClient(client=openai_client, model=model)

runner = OpenAIResponsesRunner(
    tools=tools_obj,
    developer_prompt=system_prompt,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [158]:
runner.run();

You: create a browser based vanilla js snake game in snake.html


-> Response received


-> Response received


-> Response received


You: stop


Chat ended.
